# 03 – Features: does the envelope spectrum point to the damaged part?

Phase 2. `bearing_monitor.features` reduces every snapshot and channel to 23 features (time domain,
Welch band powers, envelope spectrum). This notebook checks the features against the data: the
envelope spectrum of a failed bearing should show lines at the fault frequency of its defect.

Requires `python scripts/compute_features.py data/raw` (about two minutes on four cores) and the raw
files of test 2 for the spectra. The notebook is committed without outputs; the figure is in
`docs/figures/envelope_features.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bearing_monitor import features, io
from bearing_monitor.config import FIGURES_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR

RUN_IN, TRAIN = 200, 250
FAILED = {(1, 3): "inner race", (1, 4): "roller", (2, 1): "outer race", (3, 3): "outer race"}
FAULT_COLORS = {"bpfo": "tab:red", "bpfi": "tab:blue", "bsf": "tab:green"}

tables = {}
for test in (1, 2, 3):
    table = pd.read_parquet(PROCESSED_DATA_DIR / f"features_test{test}.parquet")
    tables[test] = table[~table["flatline"]]
{k: round(v, 1) for k, v in features.FAULT_FREQUENCIES_HZ.items()}

## Envelope spectrum of test 2, bearing 1 (outer race failure)

A snapshot from the training window (healthy) against one from the damaged end of the run. Dashed
lines: theoretical BPFO, BPFI and BSF with two harmonics; shaded: the search windows of the
envelope features.

In [ ]:
test_dir = next(d for d in io.find_test_dirs(RAW_DATA_DIR) if "2nd_test" in d.parts)
snapshots = io.list_snapshots(test_dir)
examples = {"healthy (snapshot 300)": 300, "damaged (snapshot 700)": 700}

fig = plt.figure(figsize=(16, 8.5), layout="constrained")
grid = fig.add_gridspec(2, 4)
top = [fig.add_subplot(grid[0, :2]), fig.add_subplot(grid[0, 2:])]
for ax, (label, position) in zip(top, examples.items(), strict=True):
    x = io.bearing_signal(io.load_snapshot(snapshots.iloc[position], 4), 2, 1)
    freqs, amplitude = features.envelope_spectrum(x)
    keep = freqs <= 1_000
    ax.plot(freqs[keep], amplitude[keep], color="0.2", lw=0.7)
    below, above = features.ENVELOPE_TOLERANCE
    for name, f0 in features.FAULT_FREQUENCIES_HZ.items():
        for h in range(1, features.ENVELOPE_HARMONICS + 1):
            ax.axvline(
                h * f0,
                color=FAULT_COLORS[name],
                ls="--",
                lw=0.8,
                label=name.upper() if h == 1 else None,
            )
            ax.axvspan(
                h * f0 * (1 - below),
                h * f0 * (1 + above),
                color=FAULT_COLORS[name],
                alpha=0.12,
                lw=0,
            )
    peak = freqs[keep][np.argmax(amplitude[keep][freqs[keep] > 20]) + np.sum(freqs[keep] <= 20)]
    ax.set_title(f"Test 2, bearing 1, {label}: largest line at {peak:.1f} Hz")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Envelope amplitude")
    ax.legend(loc="upper right")

for col, ((test, bearing), defect) in enumerate(FAILED.items()):
    ax = fig.add_subplot(grid[1, col])
    table = tables[test]
    sensors = len(io.TEST_CONFIGS[test].bearing_channels[bearing])
    for name, color in FAULT_COLORS.items():
        values = pd.concat(
            [
                features.bearing_features(table, bearing, s)[f"env_{name}_energy"]
                for s in range(sensors)
            ],
            axis=1,
        ).max(axis=1)
        smooth = values.rolling(6, min_periods=1).median()
        ax.semilogy(table["file_index"], smooth, color=color, lw=0.9, label=name.upper())
    ax.axvspan(RUN_IN, RUN_IN + TRAIN, color="tab:green", alpha=0.08, lw=0)
    ax.set_title(f"Test {test}, bearing {bearing} ({defect})")
    ax.set_xlabel("Snapshot")
    if col == 0:
        ax.set_ylabel("Envelope energy (1 h median)")
        ax.legend(loc="upper left")
fig.savefig(FIGURES_DIR / "envelope_features.png", dpi=120)

## Onsets per feature

The descriptive onset rule of phase 1 (mean + 6 std of positions 200 to 449, trailing median of 6,
sustained for 6 snapshots) applied to every feature of the failed bearings. Like in phase 1 this is
a look at the data, not a detector; for test 1 the larger of the two sensors' onsets is not taken,
both are listed.

In [ ]:
from bearing_monitor import trends

rows = []
for (test, bearing), defect in FAILED.items():
    table = tables[test]
    for sensor in range(len(io.TEST_CONFIGS[test].bearing_channels[bearing])):
        one = features.bearing_features(table, bearing, sensor).reset_index(drop=True)
        row = {"test": test, "bearing": bearing, "sensor": sensor, "defect": defect}
        for name in features.FEATURE_NAMES:
            row[name] = trends.first_sustained_exceedance(one[name], slice(RUN_IN, RUN_IN + TRAIN))
        rows.append(row)
onsets = pd.DataFrame(rows).set_index(["test", "bearing", "sensor", "defect"])
onsets.T